# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Musadiq8699/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Summary of Key Field Distributions

* **Traffic Volume (`clicks_last_30d`, `impressions_last_30d`):** Exhibits extreme right-skewness (heavy tail). While the median click count per page is low, top-tier pages pull tens of thousands of clicks, making the standard mean highly unrepresentative.
* **Decay Signals (`click_decay_ratio`, `impression_decay_ratio`):** Centered around 1.0 (stable traffic), but shows a heavy left tail of pages near 0.0 (severe traffic decomposition) and an upper tail of newly viral pages.
* **SERP Position (`position_last_30d`):** Skewed toward higher numerical rank positions (>50) for the long tail of low-performing pages, with median position around 45–60.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import duckdb
from huggingface_hub import HfApi, hf_hub_download
from google.colab import userdata

# ==============================================================================
# STEP 1: FETCH DATA VIA DUCKDB & PREPARE FEATURE VECTOR
# ==============================================================================

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

api = HfApi()
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
fact_files = [f for f in repo_files if f.startswith("fact_content_daily_performance/") and f.endswith(".parquet")]

local_paths = [
    hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=HF_TOKEN)
    for f in fact_files
]

con = duckdb.connect()
con.execute(f"CREATE VIEW fact_table AS SELECT * FROM read_parquet({local_paths})")

query = """
WITH max_date_cte AS (SELECT MAX(report_date) AS max_date FROM fact_table),
page_aggregates AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,
        SUM(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_impressions ELSE 0 END) AS impressions_last_30d,
        AVG(CASE WHEN report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '30 days' THEN gsc_avg_position ELSE NULL END) AS position_last_30d,
        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days' THEN gsc_clicks ELSE 0 END) AS clicks_prev_30d,
        SUM(CASE WHEN report_date < (SELECT max_date FROM max_date_cte) - INTERVAL '30 days'
                  AND report_date >= (SELECT max_date FROM max_date_cte) - INTERVAL '60 days' THEN gsc_impressions ELSE 0 END) AS impressions_prev_30d
    FROM fact_table
    GROUP BY content_hash_id
)
SELECT * FROM page_aggregates;
"""

df = con.execute(query).df()

# Calculate ratios
df['click_decay_ratio'] = (df['clicks_last_30d'] / (df['clicks_prev_30d'] + 1e-5)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
df['impression_decay_ratio'] = (df['impressions_last_30d'] / (df['impressions_prev_30d'] + 1e-5)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
df['position_last_30d'] = df['position_last_30d'].fillna(df['position_last_30d'].median())


# ==============================================================================
# STEP 2: AUDIT DISTRIBUTIONS & HEAVY TAILS (PERCENTILES)
# ==============================================================================

audit_cols = ['clicks_last_30d', 'impressions_last_30d', 'position_last_30d', 'click_decay_ratio', 'impression_decay_ratio']

# Compute key percentiles to observe heavy tails
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
dist_summary = df[audit_cols].describe(percentiles=percentiles).T

print("=== DISTRIBUTIONS & HEAVY-TAIL AUDIT ===")
print(dist_summary[['count', 'mean', 'std', 'min', '50%', '90%', '99%', 'max']])

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 1.45MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DISTRIBUTIONS & HEAVY-TAIL AUDIT ===
                           count          mean           std  min        50%  \
clicks_last_30d         427292.0  2.910787e+00  3.200183e+02  0.0   0.000000   
impressions_last_30d    427292.0  5.265400e+02  3.738492e+03  0.0   0.000000   
position_last_30d       427292.0  1.741413e+01  1.680259e+01  0.0  12.384253   
click_decay_ratio       427292.0  1.071185e+05  3.195594e+07  0.0   0.000000   
impression_decay_ratio  427292.0  2.083674e+06  7.416460e+07  0.0   0.000000   

                               90%           99%           max  
clicks_last_30d           3.000000  4.000000e+01  1.521700e+05  
impressions_last_30d    758.000000  9.901180e+03  6.187990e+05  
position_last_30d        38.746282  8.075310e+01  5.790000e+02  
click_decay_ratio         1.666661  4.000000e+05  1.521700e+10  
impression_decay_ratio    2.999970  2.690000e+07  2.835050e+10  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.